In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
print("libraries importées !")

libraries importées !


In [2]:
df = pd.read_csv(r"C:\PFE\dataset\Data\FINAL_CSV\train70_reduced.csv", low_memory=False)
print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"Classes : {df["target"].value_counts().to_dict()}")

Dataset chargé : 231646 lignes, 34 colonnes
Classes : {'legitimate': 115824, 'dos': 91156, 'bruteforce': 10150, 'malformed': 7646, 'slowite': 6441, 'flood': 429}


In [3]:
le = LabelEncoder()
df["target_encoded"] = le.fit_transform(df["target"])

print("Encodage taget :")
for i, classe in enumerate(le.classes_):
    print(f" {classe} -> {i}")

Encodage taget :
 bruteforce -> 0
 dos -> 1
 flood -> 2
 legitimate -> 3
 malformed -> 4
 slowite -> 5


In [4]:
X = df.drop(["target", "target_encoded"], axis=1)
y = df["target_encoded"]

colonnes_texte = ['tcp.flags', 'mqtt.conack.flags', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.msg', 'mqtt.protoname']

for col in colonnes_texte:
    le_col = LabelEncoder()
    X[col] = le_col.fit_transform(X[col].astype(str))

print(f" X shape : {X.shape}")
print(f" y shape : {y.shape}")
print(f" Colonnes texte encodées !")

 X shape : (231646, 33)
 y shape : (231646,)
 Colonnes texte encodées !


In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f" Normalisation effectuée !")
print(f" Train : {X_train.shape[0]} lignes")
print(f" Test  : {X_test.shape[0]} lignes")

 Normalisation effectuée !
 Train : 185316 lignes
 Test  : 46330 lignes


In [6]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
import time

# Hyperparamètres à tester
params = {
    "n_estimators" : [100, 200, 300],
    "max_depth" : [3, 5, 7],
    "learning_rate" : [0.01, 0.1, 0.3],
    "subsample": [0.8, 1.0],
}

xgb = XGBClassifier(random_state=42, eval_metric="mlogloss", use_label_encoder="False")

search = RandomizedSearchCV(
    xgb, params, 
    n_iter=10, 
    cv=3, 
    random_state=42,
    n_jobs=-1
)

print("Entraînement XGBoost + Hyperparamètres...")
start = time.time()
search.fit(X_train, y_train)
end = time.time()

print(f"Terminé en {round(end-start, 2)} secondes !")
print(f" Meilleurs paramètres : {search.best_params_}")

Entraînement XGBoost + Hyperparamètres...
Terminé en 418.09 secondes !
 Meilleurs paramètres : {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.1}


In [7]:
# Évaluation du meilleur modèle
best_xgb = search.best_estimator_

y_pred = best_xgb.predict(X_test)

print(f"Accuracy XGBoost optimisé : {round(accuracy_score(y_test, y_pred) * 100, 2)}%")
print(f"Rapport détaillé :")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy XGBoost optimisé : 94.19%
Rapport détaillé :
              precision    recall  f1-score   support

  bruteforce       0.74      0.88      0.80      1989
         dos       0.97      0.92      0.94     18241
       flood       1.00      0.42      0.59        81
  legitimate       0.94      0.99      0.96     23253
   malformed       0.93      0.58      0.72      1474
     slowite       1.00      1.00      1.00      1292

    accuracy                           0.94     46330
   macro avg       0.93      0.80      0.84     46330
weighted avg       0.94      0.94      0.94     46330



In [8]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

print(f"TensorFlow version : {tf.__version__}")

TensorFlow version : 2.20.0


In [14]:
# Créer l'Autoencoder amélioré
input_dim = X_train.shape[1]  # 33 features

# Encoder 
input_layer = Input(shape=(input_dim,))
encoded = Dense(128, activation='relu')(input_layer)
encoded = Dense(64, activation='relu')(encoded)
encoded = Dense(32, activation='relu')(encoded)
encoded = Dense(16, activation='relu')(encoded)

# Decoder
decoded = Dense(32, activation='relu')(encoded)
decoded = Dense(64, activation='relu')(decoded)
decoded = Dense(128, activation='relu')(decoded)
decoded = Dense(input_dim, activation='linear')(decoded)

# Modèle
autoencoder = Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.summary()
print(" Autoencoder amélioré créé !")

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 33)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 33)             │         4,257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,449 (118.94 KB)

 Trainable params: 30,449 (118.94 KB)

 Non-trainable params: 0 (0.00 B)

 Autoencoder amélioré créé !


In [15]:
# Entraîner seulement sur le trafic NORMAL
X_train_normal = X_train[y_train == 3]  # 3 = legitimate

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print(" Entraînement Autoencoder...")
import time
start = time.time()

history = autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=100,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

end = time.time()
print(f" Autoencoder entraîné en {round(end-start, 2)} secondes !")

 Entraînement Autoencoder...
Epoch 1/100
651/651 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0069 - val_loss: 1.6258e-04
Epoch 2/100
651/651 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 2.3905e-04 - val_loss: 5.8854e-05
Epoch 3/100
651/651 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 3.1223e-05 - val_loss: 1.4872e-05
Epoch 4/100
651/651 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 9.8059e-05 - val_loss: 1.5407e-05
Epoch 5/100
651/651 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 1.1851e-04 - val_loss: 1.5236e-05
Epoch 6/100
651/651 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 3.7771e-04 - val_loss: 3.8472e-05
Epoch 7/100
651/651 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 4.6161e-05 - val_loss: 1.6247e-05
Epoch 8/100
651/651 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 1.0969e-04 - val_loss: 1.1943e-05
Epoch 9/100
651/651 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 5.2323e-05 - val_loss: 1.3097e-05
Epoch 10/100
651/651 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 8.1907e-05 - val_loss: 2.1775e-05
Epoch 11/100
651/651 ━━

In [16]:
# Calculer l'erreur de reconstruction
X_test_pred = autoencoder.predict(X_test)
mse = np.mean(np.power(X_test - X_test_pred, 2), axis=1)

# Vraies classes en binaire
y_test_binary = (y_test != 3).astype(int)

# Trouver le meilleur seuil automatiquement
best_threshold = None
best_accuracy = 0
best_percentile = 0

for percentile in range(10, 90, 5):
    threshold = np.percentile(mse, percentile)
    y_pred_temp = (mse > threshold).astype(int)
    acc = accuracy_score(y_test_binary, y_pred_temp)
    if acc > best_accuracy:
        best_accuracy = acc
        best_threshold = threshold
        best_percentile = percentile

print(f" Meilleur seuil : {best_threshold:.6f} (percentile {best_percentile})")
print(f" Meilleure Accuracy : {round(best_accuracy * 100, 2)}%")
print("\n Rapport détaillé :")
y_pred_auto = (mse > best_threshold).astype(int)
print(classification_report(y_test_binary, y_pred_auto, target_names=['Normal', 'Anomalie']))

1448/1448 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step
 Meilleur seuil : 0.002130 (percentile 65)
 Meilleure Accuracy : 85.18%

 Rapport détaillé :
              precision    recall  f1-score   support

      Normal       0.77      1.00      0.87     23253
    Anomalie       1.00      0.70      0.83     23077

    accuracy                           0.85     46330
   macro avg       0.89      0.85      0.85     46330
weighted avg       0.89      0.85      0.85     46330



In [18]:
import pickle
autoencoder.save(r'C:\PFE\models\autoencoder.keras')
with open(r'C:\PFE\models\autoencoder_threshold.pkl', 'wb') as f:
    pickle.dump(best_threshold, f)

print("Autoencoder sauvegardé !")
print(f"Seuil sauvegardé : {best_threshold:.6f}")

Autoencoder sauvegardé !
Seuil sauvegardé : 0.002130
